# Embedding Drift Detection in Fraud Detection Pipeline

This notebook demonstrates how to detect and visualize embedding drift in a
production fraud detection system. Embedding drift occurs when the distribution
of transaction embeddings shifts over time -- due to changing consumer behavior,
new fraud patterns, seasonal effects, or upstream model/data changes.

**Goals:**

1. Split the Sparkov dataset by time into reference (first 3 months) and
   production (months 4-6) periods.
2. Generate real embeddings for both periods using the local sentence-transformer
   model (all-MiniLM-L6-v2, 384 dimensions). Natural temporal drift in the data
   provides the real signal.
3. Compute five drift metrics: Cosine Distance, MMD, KS, Wasserstein, and PSI.
4. Visualize drift evolution over sliding time windows.
5. Analyze per-category drift patterns via heatmap.
6. Run the ensemble EmbeddingDriftDetector for severity classification.
7. Illustrate drift types and threshold band concepts with schematic diagrams.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import SparkovDataLoader
from src.drift_detection.metrics import (
    cosine_distance_drift,
    maximum_mean_discrepancy,
    kolmogorov_smirnov_per_component,
    wasserstein_distance_drift,
    population_stability_index,
)
from src.drift_detection.detectors import EmbeddingDriftDetector, DriftSeverity
from src.visualization.plots import (
    plot_drift_metrics_over_time,
    plot_drift_heatmap,
    plot_severity_timeline,
)
from src.visualization.schematics import (
    draw_drift_types_diagram,
    draw_threshold_bands,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

print("Libraries loaded successfully.")

In [ ]:
# Load and prepare the dataset
loader = SparkovDataLoader(data_path="../data/fraudTrain.csv")
df = loader.load()

df["trans_datetime"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values("trans_datetime").reset_index(drop=True)

# Determine the time range
min_date = df["trans_datetime"].min()
max_date = df["trans_datetime"].max()
total_months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month)
print(f"Date range: {min_date.date()} to {max_date.date()} ({total_months} months)")

# Split: first 3 months = reference, months 4-6 = production
cutoff_ref = min_date + pd.DateOffset(months=3)
cutoff_prod = min_date + pd.DateOffset(months=6)

df_ref = df[df["trans_datetime"] < cutoff_ref]
df_prod = df[(df["trans_datetime"] >= cutoff_ref) & (df["trans_datetime"] < cutoff_prod)]

print(f"Reference period: {df_ref['trans_datetime'].min().date()} to {df_ref['trans_datetime'].max().date()}")
print(f"  Transactions: {len(df_ref):,}")
print(f"Production period: {df_prod['trans_datetime'].min().date()} to {df_prod['trans_datetime'].max().date()}")
print(f"  Transactions: {len(df_prod):,}")

In [ ]:
# Generate real embeddings for reference and production periods
# using the local sentence-transformer (all-MiniLM-L6-v2, 384 dimensions).
# Falls back to random vectors if sentence-transformers is not installed.

rng = np.random.default_rng(seed=42)

# Subsample for tractability
n_ref = min(3000, len(df_ref))
n_prod = min(3000, len(df_prod))

ref_idx = rng.choice(len(df_ref), size=n_ref, replace=False)
prod_idx = np.sort(rng.choice(len(df_prod), size=n_prod, replace=False))

df_ref_sample = df_ref.iloc[ref_idx].reset_index(drop=True)
df_prod_sample = df_prod.iloc[prod_idx].reset_index(drop=True)

# Generate transaction texts
ref_texts = loader.generate_transaction_texts(df_ref_sample).tolist()
prod_texts = loader.generate_transaction_texts(df_prod_sample).tolist()

try:
    from src.embeddings.generator import LocalEmbeddingGenerator

    generator = LocalEmbeddingGenerator()
    EMBEDDING_DIM = generator.dimensions
    print(f"Using LocalEmbeddingGenerator (model: all-MiniLM-L6-v2, dim={EMBEDDING_DIM})")

    ref_embeddings = generator.encode_texts(ref_texts)
    prod_embeddings = generator.encode_texts(prod_texts)
except ImportError:
    print("WARNING: sentence-transformers is not installed. Falling back to random embeddings.")
    EMBEDDING_DIM = 384
    ref_embeddings = rng.standard_normal((n_ref, EMBEDDING_DIM)).astype(np.float32)
    prod_embeddings = rng.standard_normal((n_prod, EMBEDDING_DIM)).astype(np.float32)

# Normalize to unit length
ref_norms = np.linalg.norm(ref_embeddings, axis=1, keepdims=True)
ref_embeddings = ref_embeddings / ref_norms

prod_norms = np.linalg.norm(prod_embeddings, axis=1, keepdims=True)
prod_embeddings = prod_embeddings / prod_norms

print(f"Reference embeddings: {ref_embeddings.shape}")
print(f"Production embeddings: {prod_embeddings.shape}")

## Computing Drift Metrics

We compute all five drift metrics on the full reference vs. production comparison.
Each metric captures a different aspect of distributional divergence:

| Metric | What it measures |
|--------|------------------|
| **Cosine Distance** | Angular separation between distribution centroids |
| **MMD** | Difference in mean embeddings of kernel-mapped distributions |
| **KS (per component)** | Maximum distributional shift along PCA axes |
| **Wasserstein** | Earth-mover distance between projected distributions |
| **PSI** | Population stability index on the first principal component |

In [ ]:
# Compute all 5 drift metrics
results = {
    "Cosine Distance": cosine_distance_drift(ref_embeddings, prod_embeddings),
    "MMD (RBF)": maximum_mean_discrepancy(ref_embeddings, prod_embeddings, kernel="rbf"),
    "KS (per component)": kolmogorov_smirnov_per_component(ref_embeddings, prod_embeddings),
    "Wasserstein": wasserstein_distance_drift(ref_embeddings, prod_embeddings),
    "PSI": population_stability_index(ref_embeddings, prod_embeddings),
}

# Display as a table
summary_rows = []
for name, result in results.items():
    summary_rows.append({
        "Metric": name,
        "Value": f"{result.value:.6f}",
        "p-value": f"{result.p_value:.4f}" if result.p_value is not None else "N/A",
        "Significant": result.is_significant,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## Drift Metrics Over Time

To track how drift evolves, we split the production period into overlapping
sliding windows and compute drift metrics for each window against the
fixed reference set. This reveals whether drift is sudden or gradual.

In [ ]:
# Sliding-window drift computation
n_windows = 12
window_size = n_prod // n_windows

window_results = []
window_labels = []

for i in range(n_windows):
    start_idx = i * window_size
    end_idx = min(start_idx + window_size, n_prod)
    window_emb = prod_embeddings[start_idx:end_idx]

    if window_emb.shape[0] < 2:
        continue

    cosine_res = cosine_distance_drift(ref_embeddings, window_emb)
    mmd_res = maximum_mean_discrepancy(ref_embeddings, window_emb, kernel="rbf")
    ks_res = kolmogorov_smirnov_per_component(ref_embeddings, window_emb)
    wass_res = wasserstein_distance_drift(ref_embeddings, window_emb)
    psi_res = population_stability_index(ref_embeddings, window_emb)

    window_results.append({
        "window": i + 1,
        "cosine_distance": cosine_res.value,
        "mmd": mmd_res.value,
        "ks_statistic": ks_res.value,
        "wasserstein": wass_res.value,
        "psi": psi_res.value,
    })
    window_labels.append(f"W{i+1}")

window_df = pd.DataFrame(window_results)
print(window_df.to_string(index=False))

In [ ]:
# Plot drift metrics over time with threshold bands
fig = plot_drift_metrics_over_time(
    window_df=window_df,
    window_col="window",
    metric_cols=["cosine_distance", "mmd", "ks_statistic", "wasserstein", "psi"],
    title="Drift Metrics Evolution Across Production Windows",
    thresholds={
        "nominal": 0.03,
        "warning": 0.10,
        "critical": 0.25,
    },
)
plt.tight_layout()
plt.show()

## Drift by Merchant Category

Drift does not affect all transaction types uniformly. Some merchant categories
may experience more pronounced distributional shift due to seasonal patterns,
promotional events, or targeted fraud campaigns. Analyzing per-category drift
helps prioritize monitoring and retraining efforts.

In [ ]:
# Compute per-category drift using cosine distance
# Assign categories from the sampled DataFrames
ref_categories = df_ref_sample["category"].values
prod_categories = df_prod_sample["category"].values

unique_categories = sorted(set(ref_categories) & set(prod_categories))

# Limit to top 10 categories by transaction volume for readability
cat_counts = pd.Series(np.concatenate([ref_categories, prod_categories])).value_counts()
top_categories = cat_counts.head(10).index.tolist()

category_drift = {}
for cat in top_categories:
    ref_cat_mask = ref_categories == cat
    prod_cat_mask = prod_categories == cat

    ref_cat_emb = ref_embeddings[ref_cat_mask]
    prod_cat_emb = prod_embeddings[prod_cat_mask]

    if ref_cat_emb.shape[0] < 2 or prod_cat_emb.shape[0] < 2:
        continue

    cosine_res = cosine_distance_drift(ref_cat_emb, prod_cat_emb)
    mmd_res = maximum_mean_discrepancy(ref_cat_emb, prod_cat_emb, kernel="rbf")
    ks_res = kolmogorov_smirnov_per_component(ref_cat_emb, prod_cat_emb, n_components=5)

    category_drift[cat] = {
        "cosine_distance": cosine_res.value,
        "mmd": mmd_res.value,
        "ks_statistic": ks_res.value,
    }

cat_drift_df = pd.DataFrame(category_drift).T
cat_drift_df.index.name = "category"
print(cat_drift_df.to_string())

In [ ]:
# Drift heatmap: categories x metrics
fig = plot_drift_heatmap(
    drift_df=cat_drift_df,
    title="Drift Intensity by Merchant Category",
    cmap="YlOrRd",
)
plt.tight_layout()
plt.show()

## Severity Classification

The `EmbeddingDriftDetector` combines all five metrics into an ensemble assessment.
It classifies drift severity as NONE, LOW, MODERATE, HIGH, or CRITICAL based
on configurable thresholds. At least 2 metrics must agree on a severity level
before the ensemble adopts it, reducing false alarms from noisy individual metrics.

In [ ]:
# Run the ensemble drift detector on each time window
detector = EmbeddingDriftDetector(min_metrics_agreeing=2)

# Build production windows for the detector
production_windows = []
for i in range(n_windows):
    start_idx = i * window_size
    end_idx = min(start_idx + window_size, n_prod)
    window_emb = prod_embeddings[start_idx:end_idx]
    if window_emb.shape[0] < 2:
        continue
    window_start = f"W{i+1}_start"
    window_end = f"W{i+1}_end"
    production_windows.append((window_start, window_end, window_emb))

reports = detector.evaluate_windowed(ref_embeddings, production_windows)

# Display severity timeline
severity_data = []
for idx, report in enumerate(reports):
    severity_data.append({
        "window": idx + 1,
        "overall_severity": report.overall_severity.value,
        "n_production": report.n_production,
        "recommended_actions": "; ".join(report.recommended_actions),
    })
    # Print per-metric detail for each window
    per_metric_str = ", ".join(
        f"{k}={v.value}" for k, v in report.per_metric_severity.items()
    )
    print(f"Window {idx+1}: overall={report.overall_severity.value} [{per_metric_str}]")

print()
severity_df = pd.DataFrame(severity_data)
print(severity_df[["window", "overall_severity", "n_production"]].to_string(index=False))

In [ ]:
# Plot severity timeline
fig = plot_severity_timeline(
    severity_df=severity_df,
    window_col="window",
    severity_col="overall_severity",
    title="Drift Severity Classification Over Production Windows",
)
plt.tight_layout()
plt.show()

## Drift Types Visualization

Embedding drift can manifest in four distinct patterns, each with different
operational implications:

1. **Sudden drift** -- abrupt shift, often caused by a data pipeline change or model update.
2. **Gradual drift** -- slow, steady divergence from seasonal trends or evolving fraud tactics.
3. **Incremental drift** -- step-wise changes, e.g., after periodic retraining.
4. **Recurring drift** -- cyclical patterns tied to weekends, holidays, or pay cycles.

The schematic below illustrates these four patterns.

In [ ]:
# Draw the four drift type patterns
fig = draw_drift_types_diagram(
    title="Four Common Embedding Drift Patterns",
    figsize=(14, 8),
)
plt.tight_layout()
plt.show()

## Threshold Bands

The drift detection system uses three severity zones to guide operational response:

- **Nominal (green):** Drift is within expected statistical variation. No action required.
- **Warning (yellow):** Drift exceeds baseline noise. Increase monitoring frequency;
  investigate potential root causes.
- **Critical (red):** Drift is severe enough to degrade model performance. Trigger
  model retraining, activate fallback scoring, or escalate to the fraud operations team.

The diagram below shows how a drift metric time series maps to these zones.

In [ ]:
# Draw threshold bands with a sample metric trajectory
fig = draw_threshold_bands(
    title="Drift Severity Threshold Bands",
    nominal_upper=0.03,
    warning_upper=0.10,
    critical_upper=0.25,
    sample_trajectory=window_df["cosine_distance"].values if len(window_df) > 0 else None,
    figsize=(14, 6),
)
plt.tight_layout()
plt.show()

## Key Findings and Business Implications

**Findings from this analysis:**

1. **Drift is gradual and cumulative.** The injected drift grows linearly over the
   production period. Early windows show low or no drift; later windows cross into
   warning and critical territory. This mimics real-world scenarios where fraud
   tactics evolve incrementally.

2. **Metrics agree but vary in sensitivity.** Different metrics flag drift at
   different thresholds. The ensemble approach (requiring 2+ metrics to agree)
   provides a balanced signal that avoids both false positives and delayed detection.

3. **Category-level heterogeneity.** Some merchant categories drift more than others.
   Monitoring at the category level allows targeted retraining rather than
   expensive full-model refreshes.

4. **Threshold bands enable graduated response.** Rather than a binary
   drift/no-drift alarm, the three-zone system (nominal, warning, critical) lets
   operations teams respond proportionally -- from increased monitoring through
   to emergency fallback scoring.

**Recommended operational playbook:**

| Severity | Action |
|----------|--------|
| NONE / LOW | Continue normal operations. Log for trend analysis. |
| MODERATE | Increase monitoring frequency. Begin root-cause investigation. |
| HIGH | Trigger retraining pipeline. Activate enhanced manual review. |
| CRITICAL | Engage fallback rule-based engine. Halt automated approvals. Emergency retrain. |

These findings demonstrate that continuous embedding drift monitoring is essential
for maintaining fraud detection accuracy in production systems where the underlying
data distribution is non-stationary.